Mandatory assignement 

1a: Recover the private key and plaintext

Brute force approach:

In [5]:

# public parameters:
p = 29837
g = 42
pk = 22690 

# c1 and c2 has been itercepted:
c1 = 23447
c2 = 8372

def recover(p, c1, c2):

    # find exponent x - the private key 
    for i in range(p-1):
        if (pow(g, i, p) == pk):
            x = i
            break
    
    # find m
    c1_ = pow(c1,-x,p) # -x is inverse
    m = (c2*c1_) % p # Modular division means multiplying by an inverse
    
    return m

print(recover(p, c1, c2))

26000


**1b. Modify the ciphertext without the private key**

The fictional target identifier is m' = 12345.

Goal is to modify c2 that the message become: 12345


**Step 1**: 
We know from the nature of ElGamal that c2 = m · PK^r mod p.

**Step 2**: 
I want to define a new c2' still using PK^r, but with, m' in stead of m. That is:
c2' ≡ m' * PK^r (mod p)

**Step 3**: 
Here I find a multiplier k

Say c2' = k * c2. 

Then
k * m * PK^r ≡ m' * PK^r (mod p)

PK^r cancels from both sides:

k * m ≡ m' (mod p)

k ≡ m' * m (mod p) #Modular division means multiplying by an inverse

Now, k only depends on m, m' and p, which are all allowed inputs.


In [6]:
m = 26000
m_target = 12345 # target identifier

# change the cyphertext to 12345
def attack(p, c1, c2, m, m_target):
    m_inverse = pow(m, -1, p)   # modular inverse of m,
    k = (m_target * m_inverse) % p  # multply by the target
    c2_target = (k * c2) % p
    return c1, c2_target

c1_, c2_ = attack(p, c1, c2, m, m_target)

# Verification using the private key x from 1a:
print(recover(p, c1_, c2_))



12345


1b: 
* Give modified cipertext
* justify your construction algebraically
* may use the recovered private key afterwards to verify the result

reflection: 
(elgamal beskytter ikke man in the middle attack)


**Exercise 2: Hash me if you can**

Context:
* UTF-8 is an "encryption" for turning characters into bytes 
* Unicode Transformation Format 8-bit
* text -> bytes

In [7]:
from hashlib import sha256

# getting to know the 3 versions of digest and sha256

# adding bits changes the length of the truncated digest 
digest = sha256("hello".encode("utf-8")).digest()  # 32 bytes
h16 = digest[:2]   # first 2 bytes / 16 bits
h24 = digest[:3]   # first 3 bytes / 24 bits
h32 = digest[:4]   # first 4 bytes / 32 bits

print(h16.hex())  # 2cf2
print(h24.hex())  # 2cf24d
print(h32.hex())  # 2cf24dba

# Changing a single character scrambles the output completely
input1 = sha256("johan".encode("utf-8")).digest()  # 32 bytes
input2 = sha256("johann".encode("utf-8")).digest()  # 32 bytes

print((input1[:4]).hex()) # d9a31550
print((input2[:4]).hex()) # a27d6fe2





2cf2
2cf24d
2cf24dba
d9a31550
a27d6fe2


**2a** Implement a collision search:

For each output length, find distinct messages with the same truncated digest

In [8]:
def find_collision(num_bytes):
    messages = {} #dictionary of truncated hash and messages
    i = 0

    while True:
        i += 1
        newMsg = "msg" + str(i)
        h = sha256(newMsg.encode("utf-8")).digest()[:num_bytes]
        if h in messages:
            return i, newMsg, messages[h]
        else:
            messages[h] = newMsg

# Test with the three levels of digest bits:
for n in (2, 3, 4):
    searches, m1, m2 = find_collision(n)
    print (
        n, "bits:", 
        m1, m2,
        "searches:", searches)

2 bits: msg121 msg114 searches: 121
3 bits: msg1576 msg917 searches: 1576
4 bits: msg151340 msg39097 searches: 151340


**2b** Distinguish collision and second-preimage resistance. 

* How does the task change if m1 is fixed in advance and you must find a different m2 with the same hash? 

Instead of finding any two messages that collide, you must match one specific, already-chosen hash value H_b(m₁). The birthday effect disappears: you can't compare new messages against each other, only against that single target. Each new message m₂ is an independent trial that succeeds with probability 1/2^b, so you hash candidates one at a time and check whether each equals the target, with no benefit from storing earlier results.

| Setting | Expected work | b = 16 | b = 24 | b = 32 |
|---|---|---|---|---|
| Collision (any pair) | ≈ 2^(b/2) | ~2⁸ | ~2¹² | ~2¹⁶ |
| fixed m₁ | ≈ 2^b | 2¹⁶ | 2²⁴ | 2³² |

* Compare the expected work for generic attacks in the two settings. 
(You do not need to implement a second-preimage search)

* Does your experiment show that full SHA-256 is practically broken?

The second-preimage attack is 2^(b/2) times more expensive. For b = 32, that is roughly 65 thousand hashes versus about 4.3 billion. The collision attack also needs memory to store about 2^(b/2) hashes, while the second-preimage search needs essentially none.

For full SHA-256 (b = 256) the gap is 2¹²⁸ for collisions versus 2²⁵⁶ for second preimages, both far out of reach, which supports your answer that the experiment does not show SHA-256 is broken.


**Exercise 3: Breaking BAD** 

Tip: 2nd exercise, consider the rotation property

First, definitions of some of the concepts that are new to me:

* **Block cypher:** A block cipher is an encryption that works on data in fixed-size chunks (blocks), rather than one bit or byte at a time. It takes one block of plaintext plus a secret key and produces one block of ciphertext of the same size.

One BAD block is 128 bits.

* **Big endian integer:** means first byte is the most significant

**3A** Derive decryption. Give a formula for decryption when both keys are known, and explain why it inverts encryption

The encryption follows this formular: 
$ E_{K_1,K_2(P)} = R(P \oplus K_1) \oplus K_2 $ ,


The encryption takes place in three steps: 
* 1: $ x = P \oplus K1 $
* 2: y = Rotate x 13 bits (left)
* 3: $ C = Y \oplus K2 $

To decrypt, I have inverted the sequence and done the invers of each step:
* 3: $ y = C \oplus K_2 $ (XOR is its own invers)
* 2: x = Rotate y 13 bits (right)
* 1: $ P = x \oplus K_1 $

The combined formular is: 
$ D_{K_1,K_2}(C) = R^{-1}(C \oplus K_2) \oplus K_1 $

** 3B Recover the challenge plaintexts.** 
* Find an attack using the known plaintext/ciphertext pair. 

* Write code that recovers all three challenge plaintexts without exhaustive key search. 

* Report the recovered bytes in hexadecimal and as readable ASCII. 

* Explain why the attack works for any plaintext block encrypted under these same keys.

In [9]:
# helper method for rotation:
def rot_l(x, n = 13, bits=128):
    n%= bits
    return ((x << n) | (x >> (bits - n))) & ((1 << bits) - 1)

def rot_r(x, n = 13, bits=128):
    n %= bits
    return ((x >> n) | (x << (bits - n))) & ((1 << bits) - 1)

# helpers for translation:
def hex_to_int(h):
    return int.from_bytes(bytes.fromhex(h), "big")

def int_to_byte(x):
    return x.to_bytes(16, "big")

''' def decrypt(c, k1, k2):
    return k1^(rot_r(c ^k2)) '''

' def decrypt(c, k1, k2):\n    return k1^(rot_r(c ^k2)) '

In [18]:
# known_plaintext_hex": 
known_p_hex = "6b6e6f776e2d626c6f636b2d30303031"
known_c_hex = "0e2061c487fe8e09eda7238088d1a6ea"
  
#challenges c1, c2, c3:
c1_c_hex = "4d6200ace534a5a346c483828ad385e9"
c2_c_hex = "8fe200a68d9547e184c681828ad38769"
c3_c_hex = "6d4381ef669d6f49076f48628ad387a9"

target_p_hex = "73747564656e743d3132333435202020"

# Effective key: K = R(K1) ^ K2 = C0 ^ R(P0)
p0 = hex_to_int(known_p_hex)
c0 = hex_to_int(known_c_hex)
k_eff = c0 ^ rot_l(p0)

def decrypt(c):
    p = int_to_byte(rot_r(hex_to_int(c) ^ k_eff))
    print(p.hex())
    print(p.decode("ascii"))

for i in (c1_c_hex, c2_c_hex, c3_c_hex):
    decrypt(i)

73747564656e743d3236303030202020
student=26000   
67726164653d31322020202020202020
grade=12        
617574686f72697365643d6e6f202020
authorised=no   


4:
use filter and time, to construct what is happening